In [1]:
import json, pandas as pd
from pathlib import Path
R=Path('results') if Path('results').exists() else Path('experiments/E13_gpt_context_architecture/results')
d=json.load(open(R/'e13_analysis.json')); S=d['summary']; K=['full','rag']; sens=json.load(open(R/'e13_evidence_mapping_sensitivity.json')); man=json.load(open(R/'e13_manual_inspection.json'))
pd.set_option('display.width',220,'display.max_colwidth',100)

# E13 — GPT full-context vs RAG (DEV_ARCH_v1)
**DEV architecture validation with disclosed historical exposure** (DEV is not unseen/pristine; E00–E12C never used DEV; TEST untouched).
## 1. Research question
Under the same GPT-5-mini, GPT-P0, output contract and evaluator, does retrieval_v1 RAG outperform or adequately match full-NDA context while reducing tokens, latency and cost?

## 2. DEV exposure disclosure
Historical T-series work touched 58/61 DEV docs (150-case seed-42 sample) plus 55 golden cases; DEV_ARCH_v1 is case-level disjoint from both; 48/51 of its documents were touched by other hypotheses.

## 3. Manifest

In [2]:
print(json.load(open(R/'dev_audit.json'))['manifest_summary'] | {}) 

{'manifest_id': 'DEV_ARCH_v1', 'purpose': 'E13 GPT full-context vs RAG architecture validation (DEV = Role B)', 'source_split': 'dev', 'seed': 1400, 'target_per_class': 50, 'max_cases_per_doc_per_class': 2, 'sampling_algorithm': 'Pool = DEV cases not in the historical 150-case seed-42 sample and not in any data/golden case (case-level exclusion; document-level exclusion infeasible). Per class: sort doc ids, shuffle with random.Random(1400), take <= 2 cases/doc until 50. Sorted by (class, doc, hyp). Performance-agnostic; no hard-case enrichment.', 'total_cases': 150, 'per_class_case_counts': {'Contradiction': 50, 'Entailment': 50, 'NotMentioned': 50}, 'total_unique_documents': 51, 'selected_document_ids': [7, 9, 13, 14, 15, 16, 17, 19, 20, 27, 28, 32, 33, 35, 37, 39, 44, 45, 54, 56, 63, 64, 69, 70, 71, 72, 73, 75, 79, 81, 82, 152, 205, 398, 405, 411, 435, 440, 456, 478, 482, 486, 507, 547, 563, 575, 590, 595, 598, 605, 610], 'documents_also_touched_by_historical_sample_or_golden': 48, '

## 4. Frozen architecture setup
FULL: complete NDA text; RAG: retrieval_v1 top-5 (BM25→clause_256→top-20→MiniLM-L-12→top-5). Same GPT-5-mini, GPT-P0 (sha1 3fcc7c95…), schema/parser/validator/scorer, temperature 0, 60s timeout, concurrency 5, FULL then RAG.

## 5. Common prompt template
System: byte-identical GPT-P0. User: `Requirement: {hypothesis_text}\n\nNDA context: {context_text}`. Only the context text differs. Limitation: the system prompt says 'excerpts retrieved from the NDA', not natural for FULL.

## 6. Pre-run context/token analysis

In [3]:
f=json.load(open(R/'pre_run_forecast.json')); print({k:{a:round(b) for a,b in v.items()} for k,v in f['request_tokens_calibrated'].items()}); print(f['reduction_full_to_rag']); print(f['context_window_safety'])

{'full': {'mean': 2457, 'median': 2259, 'p90': 4416, 'p95': 5826, 'max': 6323}, 'rag': {'mean': 1136, 'median': 1138, 'p90': 1278, 'p95': 1312, 'max': 1521}}
{'mean_abs': 1320.3811874937292, 'mean_pct': 53.74688812298031, 'median_pct': 49.60526315789473, 'nda_text_mean_pct': 56.939843118503596}
{'provider_listing_context_length': 400000, 'provider_max_completion_tokens': 128000, 'source': "OpenRouter public GET /api/v1/models (metadata only, no model call), 2026-09-27; also matches E12A's assumed 400K", 'repo_config_records_context_window': False, 'max_full_request_tokens_cl100k': 6382, 'max_full_request_tokens_calibrated': 6382, 'headroom_pct_vs_400k': 97.6545, 'would_truncate_or_reject_any_case': False}


## 7. Execution integrity

In [4]:
print(json.load(open(R/'pre_run_verification.json'))['checks']); print(json.dumps(d['run'],indent=1)[:1600])

{'manifest_DEV_ARCH_v1_seed1400_150_51docs': True, 'full_artifact_same_ids_order': True, 'rag_artifact_same_ids_order': True, 'same_hypothesis_text_both_arms': True, 'full_context_is_complete_dev_document': True, 'rag_top5_only': True, 'no_evaluator_truth_in_model_facing': True, 'system_prompt_full_hash_is_gpt_p0': True, 'ledger_approx_2_2317': True, 'no_existing_e13_ledger_rows': True, 'prior_ledger_rows_e12b_600_e12c_300': True, 'no_existing_e13_outputs': True, 'budget_gate_pass': True}
{
 "arms": {
  "full": {
   "spend_usd": 0.31196474999999996,
   "wall_seconds": 232.29685916600283,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_calls": 0,
   "verify": {
    "records_150": true,
    "no_duplicates": true,
    "all_case_ids_present": true,
    "all_successful": true,
    "prompt_hash_ok": true,
    "run_id_ok": true
   }
  },
  "rag": {
   "spend_usd": 0.26070175,
   "wall_seconds": 239.02013308403548,
   "rate_limit_like_errors": 0,
   "stopped": null,
   "failed_c

## 8. FULL metrics

In [5]:
s=S['full']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall']); print('joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors'],'retries',s['retries']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.8333, 'macro_f1': 0.834, 'evidence_recall': 0.9, 'evidence_precision': 0.8571, 'source_valid_evidence_rate': 0.9143, 'joint': 0.7733}
recall {'Entailment': 0.86, 'Contradiction': 0.84, 'NotMentioned': 0.8}
joint_by_class {'Entailment': 0.78, 'Contradiction': 0.74, 'NotMentioned': 0.8}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0 retries 0
confusion (rows=gold E,C,NM) [[43, 3, 4], [7, 42, 1], [6, 4, 40]]


## 9. RAG metrics

In [6]:
s=S['rag']; print({x:round(s[x],4) for x in ['accuracy','macro_f1','evidence_recall','evidence_precision','source_valid_evidence_rate','joint']}); print('recall',s['recall']); print('joint_by_class',s['joint_by_class']); print('parse',s['parse'],'errors',s['errors'],'retries',s['retries']); print('confusion (rows=gold E,C,NM)',s['confusion_matrix'])

{'accuracy': 0.8133, 'macro_f1': 0.8144, 'evidence_recall': 0.84, 'evidence_precision': 0.8155, 'source_valid_evidence_rate': 0.9417, 'joint': 0.7533}
recall {'Entailment': 0.84, 'Contradiction': 0.8, 'NotMentioned': 0.8}
joint_by_class {'Entailment': 0.74, 'Contradiction': 0.72, 'NotMentioned': 0.8}
parse {'strict': 150, 'recovered': 0, 'invalid': 0} errors 0 retries 0
confusion (rows=gold E,C,NM) [[42, 4, 4], [7, 40, 3], [8, 2, 40]]


## 10. Matched comparison

In [7]:
rows={'Accuracy':'accuracy','Macro-F1':'macro_f1','Evidence recall':'evidence_recall','Evidence precision':'evidence_precision','Source-valid evidence':'source_valid_evidence_rate','Joint':'joint'}
t=pd.DataFrame({k:{n:S[k][m] for n,m in rows.items()} for k in K})
for c in ['Entailment','Contradiction','NotMentioned']: t.loc[c+' recall']=[S[k]['recall'][c] for k in K]
for c in ['Entailment','Contradiction','NotMentioned']: t.loc['Joint '+c]=[S[k]['joint_by_class'][c] for k in K]
t.loc['Joint (cases/150)']=[S[k]['joint_n'] for k in K]; t['rag-full']=t.rag-t.full; t.round(4)

,full,rag,rag-full
Accuracy,0.8333,0.8133,-0.0200
Macro-F1,0.8340,0.8144,-0.0196
Evidence recall,0.9000,0.8400,-0.0600
Evidence precision,0.8571,0.8155,-0.0416
Source-valid evidence,0.9143,0.9417,0.0275
Joint,0.7733,0.7533,-0.0200
Entailment recall,0.8600,0.8400,-0.0200
Contradiction recall,0.8400,0.8000,-0.0400
NotMentioned recall,0.8000,0.8000,0.0000
Joint Entailment,0.7800,0.7400,-0.0400


## 11-12. Classification and joint transitions (FULL → RAG; discordant IDs)

In [8]:
p=d['paired']; print('CLS',p['cls']); print('JOINT',p['joint'])
for k in ['cls_full_wrong_rag_right','cls_full_right_rag_wrong','joint_full_fail_rag_success','joint_full_success_rag_fail']: print(k,p[k])

CLS {'wrong_to_right': 9, 'right_to_wrong': 12, 'both_right': 113, 'both_wrong': 16}
JOINT {'wrong_to_right': 13, 'right_to_wrong': 16, 'both_right': 100, 'both_wrong': 21}
cls_full_wrong_rag_right ['dev::44::nda-7', 'dev::75::nda-20', 'dev::610::nda-4', 'dev::152::nda-1', 'dev::456::nda-15', 'dev::64::nda-10', 'dev::507::nda-1', 'dev::563::nda-16', 'dev::595::nda-1']
cls_full_right_rag_wrong ['dev::56::nda-20', 'dev::440::nda-1', 'dev::478::nda-1', 'dev::547::nda-16', 'dev::563::nda-1', 'dev::7::nda-1', 'dev::73::nda-13', 'dev::547::nda-10', 'dev::13::nda-11', 'dev::547::nda-15', 'dev::563::nda-11', 'dev::598::nda-16']
joint_full_fail_rag_success ['dev::44::nda-7', 'dev::69::nda-2', 'dev::75::nda-20', 'dev::563::nda-2', 'dev::610::nda-4', 'dev::13::nda-12', 'dev::15::nda-12', 'dev::456::nda-15', 'dev::610::nda-12', 'dev::64::nda-10', 'dev::507::nda-1', 'dev::563::nda-16', 'dev::595::nda-1']
joint_full_success_rag_fail ['dev::37::nda-2', 'dev::56::nda-20', 'dev::435::nda-2', 'dev::478:

## 13. Contradiction analysis (every discordant case manually inspected)

In [9]:
print(pd.DataFrame(d['contradiction'])); print(json.dumps(man['contradiction_discordant_13'],indent=1))

                  full    rag
correct          42.00  40.00
of               50.00  50.00
recall            0.84   0.80
to_Entailment     7.00   7.00
to_NotMentioned   1.00   3.00
joint            37.00  36.00
joint_rate        0.74   0.72
{
 "RAG_worse_8": [
  "37::nda-2 evidence loss (omission)",
  "56::nda-20 label lost (unclear, gold in context)",
  "435::nda-2 partial evidence",
  "440::nda-1 label lost (omission)",
  "478::nda-1 label lost (carve-out omission)",
  "547::nda-16 label lost (variability)",
  "563::nda-1 label lost (omission)",
  "590::nda-2 evidence loss (omission)"
 ],
 "RAG_better_5": [
  "44::nda-7 (variability)",
  "69::nda-2 (mapping artifact)",
  "75::nda-20 (possible FULL distractor/variability)",
  "563::nda-2 (mapping artifact)",
  "610::nda-4 (carve-out surfaced by retrieval)"
 ],
 "net": "Contradiction correct FULL 42 vs RAG 40 (RAG lost 5, gained 3); Contradiction joint 37 vs 36"
}


## 14. Context-loss cases (FULL success → RAG failure; all 18 inspected)

In [10]:
print(json.dumps(man['context_loss_FULL_success_to_RAG_failure_18'],indent=1))

{
 "retrieval_filtering_omission (gold spans not in RAG top-5)": [
  "dev::37::nda-2 (C: label kept, evidence from wrong clauses)",
  "dev::440::nda-1 (C: RAG NotMentioned, no evidence)",
  "dev::478::nda-1 (C: 'Notwithstanding' carve-out clause missed -> Entailment)",
  "dev::563::nda-1 (C: RAG NotMentioned)",
  "dev::590::nda-2 (C: label kept, definition span missed)",
  "dev::37::nda-19 (E: label kept, gold survival clause missed)",
  "dev::73::nda-13 (E: RAG NotMentioned; longest NDA, 6.2K tokens)"
 ],
 "evidence_selection_issue (gold in RAG context, partial/alternative evidence)": [
  "dev::435::nda-2 (C: only 1 of 4 gold spans quoted)",
  "dev::37::nda-12 (E: 1 of 3 gold spans)",
  "dev::56::nda-13 (E: cited a different valid exclusion clause)"
 ],
 "context_isolation_over_inference_on_absent_topics (gold NotMentioned; RAG saw only related chunks and inferred a label)": [
  "dev::13::nda-11",
  "dev::547::nda-15",
  "dev::563::nda-11",
  "dev::598::nda-16"
 ],
 "model_variability

## 15. Distractor cases (FULL failure → RAG success; all 14 inspected)

In [11]:
print(json.dumps(man['distractor_FULL_failure_to_RAG_success_14'],indent=1))

{
 "evaluation_mapping_artifact (correct label; FULL quote merged lines/ellipsis so exact-substring span mapping failed; rescued by tolerant mapping)": [
  "dev::69::nda-2",
  "dev::563::nda-2",
  "dev::13::nda-12",
  "dev::15::nda-12",
  "dev::610::nda-12 (also partial-evidence drift)"
 ],
 "plausible_full_context_over_inference (gold NotMentioned; FULL cited related clauses -> Entailment/Contradiction; RAG returned NotMentioned)": [
  "dev::64::nda-10 (5.7K-token NDA)",
  "dev::507::nda-1",
  "dev::563::nda-16 (conditional return/destroy clause)",
  "dev::595::nda-1 (definition-clause distraction, 4.3K tokens)"
 ],
 "carve_out_surfaced_by_retrieval": [
  "dev::610::nda-4 (FULL Entailment missed the 'Residuals' exception; RAG's top-5 surfaced it -> Contradiction)"
 ],
 "model_variability_or_unclear": [
  "dev::44::nda-7 (identical evidence spans, different label)",
  "dev::456::nda-15 (442-token NDA; RAG context ~ whole document)",
  "dev::152::nda-1 (label recovered, evidence still w

### POST-HOC EVALUATOR SENSITIVITY ANALYSIS: whitespace-tolerant evidence-span mapping (NOT the frozen result; frozen headline metrics are not replaced)
Frozen scoring maps quotes to spans by exact substring; several FULL quotes merged lines and failed to map. Tolerant mapping, applied to both arms, rescues 5 FULL vs 1 RAG cases: net_joint −7, evidence recall gap −10pp, precision gap −8pp; outcome letter unchanged (B).

In [12]:
print({k:{a:b for a,b in v.items()} for k,v in sens.items() if k in ('full','rag')}); print('net_joint_tolerant',sens['net_joint_tolerant'],sens['guards_tolerant'],sens['outcome_letter_if_this_mapping_had_been_frozen'])

{'full': {'joint_strict': 116, 'joint_tolerant': 121, 'rescued_by_tolerant_mapping': ['dev::69::nda-2', 'dev::563::nda-2', 'dev::13::nda-12', 'dev::15::nda-12', 'dev::610::nda-12'], 'quotes_unmappable_strict': 4, 'quotes_unmappable_tolerant': 0, 'evidence_recall_tolerant': 0.94, 'evidence_precision_tolerant': 0.8952380952380953}, 'rag': {'joint_strict': 113, 'joint_tolerant': 114, 'rescued_by_tolerant_mapping': ['dev::435::nda-2'], 'quotes_unmappable_strict': 0, 'quotes_unmappable_tolerant': 0, 'evidence_recall_tolerant': 0.84, 'evidence_precision_tolerant': 0.8155339805825242}}
net_joint_tolerant -7 {'contradiction_delta': -2, 'macro_f1_delta': -0.01959617867106389, 'evidence_recall_delta': -0.09999999999999998, 'evidence_precision_delta': -0.07970411465557103, 'fails': {'1': False, '2': False, '3': True, '4': True}} B


## 16. Length-group analysis (predeclared cutoffs 1,352 / 2,683 FULL-NDA tokens)

In [13]:
pd.DataFrame(d['length_groups']).T.round(3)

,n,nda_tokens_mean,full_request_tokens_mean,rag_request_tokens_mean,full_accuracy,rag_accuracy,full_joint,rag_joint,full_joint_n,rag_joint_n,joint_delta_cases,accuracy_delta
short,53.0,1139.151,1270.189,1069.509,0.811,0.830,0.755,0.774,40.0,41.0,1.0,0.019
medium,49.0,2114.204,2223.633,1155.490,0.816,0.735,0.776,0.714,38.0,35.0,-3.0,-0.082
long,48.0,3903.688,3999.938,1198.667,0.875,0.875,0.792,0.771,38.0,37.0,-1.0,0.000


## 17-18. Token / cost / latency comparison (actual measurements)

In [14]:
pd.DataFrame({k:{**{'in_'+a:b for a,b in S[k]['input_tokens'].items()},**{'out_'+a:b for a,b in S[k]['output_tokens'].items()},**{'lat_'+a:b for a,b in S[k]['latency_ms'].items()},'cost_total':S[k]['cost']['total'],'cost_mean':S[k]['cost']['mean'],'per_1000':S[k]['cost_projection']['per_1000'],'per_8000':S[k]['cost_projection']['per_8000']} for k in K}).round(4)

,full,rag
in_mean,2455.1667,1138.9267
in_median,2255.5000,1139.0000
in_p90,4439.0000,1280.0000
in_p95,5738.0000,1296.0000
in_max,6253.0000,1537.0000
out_mean,732.9867,726.6400
out_median,673.5000,654.0000
out_p90,1229.0000,1236.0000
out_p95,1536.0000,1391.0000
out_max,2152.0000,2152.0000


In [15]:
d['operational']

{'input_token_ratio_full_over_rag_mean': 2.155684591925731,
 'latency_ratio_full_over_rag_mean': 0.9958160143411167,
 'latency_ratio_full_over_rag_median': 1.0181696296697433,
 'latency_ratio_p95': 1.027924988991108,
 'cost_ratio_full_over_rag': 1.19663466010489,
 'output_token_ratio_full_over_rag_mean': 1.0087342654776323,
 'input_token_reduction_pct_rag_vs_full': 53.61102437037539,
 'timeouts': {'full': 0, 'rag': 0},
 'errors': {'full': 0, 'rag': 0},
 'retries': {'full': 0, 'rag': 0}}

## 19. Paired statistics (supporting evidence only; RAG − FULL)

In [16]:
print('McNemar cls (b=FULL-only correct, c=RAG-only correct)',p['mcnemar_cls_b_fullonly_c_raglonly']); print('McNemar joint',p['mcnemar_joint']); print({a:(round(b['point'],3),[round(x,3) for x in b['ci95']]) for a,b in p['bootstrap_rag_minus_full'].items()})

McNemar cls (b=FULL-only correct, c=RAG-only correct) {'p0_only': 12, 'candidate_only': 9, 'n_discordant': 21, 'p': 0.6636238098144531}
McNemar joint {'p0_only': 16, 'candidate_only': 13, 'n_discordant': 29, 'p': 0.711071103811264}
{'accuracy': (-0.02, [-0.08, 0.04]), 'macro_f1': (-0.02, [-0.08, 0.04]), 'contradiction_recall': (-0.04, [-0.157, 0.07]), 'joint': (-0.02, [-0.087, 0.047])}


## 20. Decision-rule application (frozen)

In [17]:
print('net_joint (RAG-FULL):',d['net_joint']); print(pd.DataFrame(d['guards']).T)

net_joint (RAG-FULL): -3
                                              delta_cases  fails     delta
1_contradiction_correct_ge3_below_full                 -2  False       NaN
2_macro_f1_more_than_0.03_below_full                  NaN  False -0.019596
3_evidence_recall_more_than_3pp_below_full            NaN   True     -0.06
4_evidence_precision_more_than_3pp_below_full         NaN   True -0.041609


## 21. DEV architecture outcome

In [18]:
print(d['outcome'])

B - FULL MATERIALLY BETTER


## 22. Limitations
DEV is not unseen (disclosed exposure); one run per arm with known hosted-model variability (E12A) — a net_joint of −3 is inside the noise band and the B outcome is driven by the evidence-quality guards, not by net_joint ≤ −5; DEV NDAs are short (max request ≈6.3K tokens), so E13 does not show full context scales to enterprise-size NDAs; the frozen system prompt says 'excerpts' and is unnatural for FULL; the evidence scorer's exact-substring mapping has a whitespace artifact (sensitivity above); E13 is the DEV architecture-selection result, not the final production architecture.

**Approved interpretation:** FULL context showed a modest but consistent evidence-quality advantage over top-5 RAG on DEV; the joint difference was small and statistically uncertain, but RAG exceeded the predeclared evidence-quality guards. FULL is the preferred candidate for subsequent validation for the NDA lengths represented here; this is not a claim that FULL dominates RAG, that RAG cannot scale better, or that FULL is production-ready. retrieval_v1 is kept as a measured baseline; no hybrid router yet.